In [ ]:
import pandas as pd

# get data from the compressed file
raw_data_path = '../data/raw/listings.csv.gz'
df = pd.read_csv(raw_data_path)

print(f"Dimensions of Dataset: {df.shape[0]} rows and {df.shape[1]} columns.")

# Display the first 5 columns and 3 rows
df.iloc[:3, :5]

In [ ]:
# choose what columns to keep
selected_columns = [
    'id',
    'name',
    'neighbourhood_cleansed',
    'latitude',
    'longitude',
    'room_type',
    'accommodates',
    'bedrooms',
    'beds',
    'price',
    'minimum_nights',
    'availability_365',
    'number_of_reviews',
    'review_scores_rating',
    'reviews_per_month',
    'host_is_superhost'
]

#subset of the original array
df_subset = df[selected_columns].copy()

# check data types and null values
df_subset.info()

In [ ]:
# take a small sample to test and adjust the data
print("Δείγμα ακατέργαστων τιμών:")
print(df_subset['price'].dropna().head(5))

# clean out '$' and ',' and then convert to float
df_subset['price_cleaned'] = (
    df_subset['price']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

#print the new data with clean values
df_subset['price_cleaned'].describe()


In [ ]:
# there were some extreme values, so I will check only the upper bracket now to isolate the outliers
print("Ποσοστημόρια τιμών (95%, 98%, 99%, 99.5%):")
print(df_subset['price_cleaned'].quantile([0.95, 0.98, 0.99, 0.995]))

# keep the real values and drop the rest with dropna and filtering
df_clean = df_subset.dropna(subset=['price_cleaned']).copy()
df_clean = df_clean[df_clean['price_cleaned'] >= 15]
df_clean = df_clean[df_clean['price_cleaned'] <= 700]

# fill values for bedrooms and beds
df_clean['bedrooms'] = df_clean['bedrooms'].fillna(1)
df_clean['beds'] = df_clean['beds'].fillna(df_clean['accommodates'])

# convert column host_is_superhost to a boolean (0 ή 1)
mapped_values = df_clean['host_is_superhost'].map({'t': 1, 'f': 0})
# whatever is null we assume the host is not a superhost
filled_values = mapped_values.fillna(0)
df_clean['is_superhost'] = filled_values.astype(int)

print(f"\nOriginal rows: {len(df_subset)}")
print(f"Clean rows after filtering: {len(df_clean)}")

In [ ]:
# list of columns to export
final_columns = [
    'id',
    'name',
    'neighbourhood_cleansed',
    'latitude',
    'longitude',
    'room_type',
    'accommodates',
    'bedrooms',
    'beds',
    'price_cleaned',
    'minimum_nights',
    'availability_365',
    'number_of_reviews',
    'review_scores_rating',
    'reviews_per_month',
    'is_superhost'
]

df_export = df_clean[final_columns].copy()

df_export = df_export.rename(columns={'price_cleaned': 'price'})

output_path = '../data/processed/athens_listings_clean.csv'

df_export.to_csv(output_path, index=False)

print(f"The cleaned file was created successfully at: {output_path}")
print(f"Total records exported: {len(df_export)}")